# 🏆 **Agentic RFP Evaluation & Supplier Ranking System**
### *Classroom Mini-Project: End-to-End Agentic AI & Deterministic Procurement Scoring*

This notebook contains a complete, self-contained implementation of an AI-assisted application that:
1. Ingests supplier RFP response PDFs.
2. Evaluates proposals against configurable criteria stored in SQLite using an LLM (with evidence grounding).
3. Validates and normalizes outputs via Pydantic schemas.
4. Calculates absolute weighted scores, peer benchmarks, criterion gaps, relative percentages, and weighted Peer Performance Index (PPI).
5. Applies a strict 4-stage deterministic tie-breaker sorting algorithm.
6. Persists complete run results in SQLite and provides an interactive Streamlit UI for drill-down and JSON exports.

---

## 📦 **Step 1: Install Required Dependencies & Tunnel Tools**

In [ ]:
!pip install -q streamlit pymupdf pydantic reportlab google-genai openai pandas tabulate
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print('✅ Dependencies and Cloudflared tunnel installed successfully.')

## 🗄️ **Step 2: Create `database.py`**

In [ ]:
%%writefile database.py
import sqlite3
import json
import datetime
from typing import List, Dict, Any, Optional

DB_PATH = "rfp_eval.db"

DEFAULT_CRITERIA = [
    {
        "criterion_id": 1,
        "name": "Technical Capability",
        "description": "Architecture, integrations, scalability, technical fit",
        "weight": 30.0,
        "max_score": 10.0,
        "is_active": 1
    },
    {
        "criterion_id": 2,
        "name": "Implementation Plan",
        "description": "Timeline, milestones, staffing, risk plan",
        "weight": 20.0,
        "max_score": 10.0,
        "is_active": 1
    },
    {
        "criterion_id": 3,
        "name": "Commercial Value",
        "description": "Pricing clarity, total cost, assumptions",
        "weight": 20.0,
        "max_score": 10.0,
        "is_active": 1
    },
    {
        "criterion_id": 4,
        "name": "Security & Compliance",
        "description": "Controls, certifications, privacy, auditability",
        "weight": 20.0,
        "max_score": 10.0,
        "is_active": 1
    },
    {
        "criterion_id": 5,
        "name": "Support & Experience",
        "description": "Support model, similar projects, references",
        "weight": 10.0,
        "max_score": 10.0,
        "is_active": 1
    }
]

def get_connection(db_path: str = DB_PATH) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    return conn

def init_db(db_path: str = DB_PATH):
    """Creates SQLite tables and seeds default evaluation criteria if empty."""
    with get_connection(db_path) as conn:
        cursor = conn.cursor()
        
        # 1. evaluation_criteria table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS evaluation_criteria (
                criterion_id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                description TEXT NOT NULL,
                weight REAL NOT NULL,
                max_score REAL NOT NULL,
                is_active INTEGER NOT NULL DEFAULT 1
            )
        """)
        
        # 2. rfp_runs table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS rfp_runs (
                rfp_run_id TEXT PRIMARY KEY,
                created_at TEXT NOT NULL,
                status TEXT NOT NULL
            )
        """)
        
        # 3. supplier_results table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS supplier_results (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                rfp_run_id TEXT NOT NULL,
                supplier_name TEXT NOT NULL,
                submission_date TEXT NOT NULL,
                experience_rating REAL NOT NULL,
                absolute_score REAL NOT NULL,
                ppi REAL NOT NULL,
                final_rank INTEGER NOT NULL,
                result_json TEXT NOT NULL,
                FOREIGN KEY (rfp_run_id) REFERENCES rfp_runs (rfp_run_id)
            )
        """)
        
        # Seed default criteria if table is empty
        cursor.execute("SELECT COUNT(*) FROM evaluation_criteria")
        if cursor.fetchone()[0] == 0:
            for item in DEFAULT_CRITERIA:
                cursor.execute("""
                    INSERT INTO evaluation_criteria (criterion_id, name, description, weight, max_score, is_active)
                    VALUES (?, ?, ?, ?, ?, ?)
                """, (item["criterion_id"], item["name"], item["description"], item["weight"], item["max_score"], item["is_active"]))
        conn.commit()

def get_active_criteria(db_path: str = DB_PATH) -> List[Dict[str, Any]]:
    """Returns all active evaluation criteria sorted by criterion_id."""
    init_db(db_path)
    with get_connection(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM evaluation_criteria WHERE is_active = 1 ORDER BY criterion_id ASC")
        rows = cursor.fetchall()
        return [dict(r) for r in rows]

def get_all_criteria(db_path: str = DB_PATH) -> List[Dict[str, Any]]:
    """Returns all criteria including inactive ones."""
    init_db(db_path)
    with get_connection(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM evaluation_criteria ORDER BY criterion_id ASC")
        return [dict(r) for r in cursor.fetchall()]

def update_criterion(criterion_id: int, weight: float, max_score: float, is_active: int, db_path: str = DB_PATH):
    """Updates criterion properties."""
    with get_connection(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE evaluation_criteria
            SET weight = ?, max_score = ?, is_active = ?
            WHERE criterion_id = ?
        """, (weight, max_score, is_active, criterion_id))
        conn.commit()

def save_rfp_run(rfp_run_id: str, status: str, supplier_rankings: List[Dict[str, Any]], db_path: str = DB_PATH):
    """Saves a complete RFP run and supplier results into SQLite."""
    init_db(db_path)
    created_at = datetime.datetime.now(datetime.timezone.utc).isoformat()
    with get_connection(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO rfp_runs (rfp_run_id, created_at, status)
            VALUES (?, ?, ?)
        """, (rfp_run_id, created_at, status))
        
        for item in supplier_rankings:
            cursor.execute("""
                INSERT INTO supplier_results 
                (rfp_run_id, supplier_name, submission_date, experience_rating, absolute_score, ppi, final_rank, result_json)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                rfp_run_id,
                item["supplier_name"],
                item["submission_date"],
                float(item.get("experience_rating", 0.0)),
                float(item["absolute_score"]),
                float(item["ppi"]),
                int(item["final_rank"]),
                json.dumps(item)
            ))
        conn.commit()

def get_rfp_run(rfp_run_id: str, db_path: str = DB_PATH) -> Optional[Dict[str, Any]]:
    """Retrieves a complete RFP run by ID."""
    with get_connection(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM rfp_runs WHERE rfp_run_id = ?", (rfp_run_id,))
        run_row = cursor.fetchone()
        if not run_row:
            return None
        
        cursor.execute("SELECT * FROM supplier_results WHERE rfp_run_id = ? ORDER BY final_rank ASC", (rfp_run_id,))
        results = [json.loads(r["result_json"]) for r in cursor.fetchall()]
        
        return {
            "rfp_run_id": run_row["rfp_run_id"],
            "created_at": run_row["created_at"],
            "status": run_row["status"],
            "supplier_results": results
        }

def list_all_runs(db_path: str = DB_PATH) -> List[Dict[str, Any]]:
    """Lists all stored RFP runs."""
    init_db(db_path)
    with get_connection(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM rfp_runs ORDER BY created_at DESC")
        return [dict(r) for r in cursor.fetchall()]


## 📄 **Step 3: Create `pdf_generator.py`**

In [ ]:
%%writefile pdf_generator.py
import os
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

def generate_supplier_pdfs(output_dir: str = "."):
    """Generates 4 synthetic supplier RFP PDFs matching classroom mini-project requirements."""
    os.makedirs(output_dir, exist_ok=True)
    
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        'DocTitle',
        parent=styles['Heading1'],
        fontSize=20,
        leading=24,
        textColor=colors.HexColor('#1E3A8A'),
        spaceAfter=12
    )
    h2_style = ParagraphStyle(
        'SectionHeader',
        parent=styles['Heading2'],
        fontSize=14,
        leading=18,
        textColor=colors.HexColor('#1D4ED8'),
        spaceBefore=10,
        spaceAfter=6
    )
    body_style = ParagraphStyle(
        'BodyDark',
        parent=styles['Normal'],
        fontSize=10,
        leading=14,
        textColor=colors.HexColor('#374151')
    )

    pdf_files = {}

    # 1. Apex Systems
    fn1 = os.path.join(output_dir, "RFP_Apex_Systems.pdf")
    doc1 = SimpleDocTemplate(fn1, pagesize=letter, leftMargin=36, rightMargin=36, topMargin=36, bottomMargin=36)
    story1 = [
        Paragraph("RFP Response: Enterprise Cloud Platform Modernization", title_style),
        Paragraph("<b>Submitted By:</b> Apex Systems Inc. | <b>Date:</b> 2026-08-01", body_style),
        Spacer(1, 10),
        
        Paragraph("1. Executive Summary & Requirement Understanding", h2_style),
        Paragraph("Apex Systems offers a premier, high-security enterprise platform solution. We bring extensive engineering rigor designed for zero-downtime architecture and strict regulatory compliance.", body_style),
        Spacer(1, 8),
        
        Paragraph("2. Technical Capability & Architecture", h2_style),
        Paragraph("Our architecture leverages microservices on Kubernetes (EKS), multi-region disaster recovery, automated CI/CD pipelines, and zero-trust IAM integration. Microsecond latency APIs guarantee high throughput and 99.999% availability.", body_style),
        Spacer(1, 8),

        Paragraph("3. Implementation Plan & Milestones", h2_style),
        Paragraph("Delivery duration is 9 months structured in 4 phases: Discovery & Core Architecture (Months 1-2), System Migration (Months 3-5), Security Auditing & UAT (Months 6-7), and Final Rollout (Months 8-9). Dedicated team of 8 senior architects.", body_style),
        Spacer(1, 8),

        Paragraph("4. Commercial Value & Pricing", h2_style),
        Table([
            ["Component", "Details", "Cost (USD)"],
            ["Core Platform Software", "Enterprise License (3 Years)", "$150,000"],
            ["Implementation & Engineering", "9-Month Professional Services", "$80,000"],
            ["Annual Premium Support", "24/7 Managed Services", "$20,000"],
            ["Total Commercial Price", "Fixed Fee Contract", "$250,000"]
        ], style=[
            ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#DBEAFE')),
            ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ]),
        Spacer(1, 8),

        Paragraph("5. Security, Compliance & Risk Controls", h2_style),
        Paragraph("Certified ISO 27001, SOC 2 Type II, and HIPAA compliant. Includes automated penetration testing, AES-256 encryption at rest, TLS 1.3 in transit, and continuous SIEM monitoring.", body_style),
        Spacer(1, 8),

        Paragraph("6. Support Model & Experience", h2_style),
        Paragraph("10 years of enterprise experience with 15+ Fortune 500 reference projects. 24/7 Dedicated Tier-3 Support with 15-minute emergency SLA.", body_style),
    ]
    doc1.build(story1)
    pdf_files["Apex Systems"] = fn1

    # 2. BrightPath Tech
    fn2 = os.path.join(output_dir, "RFP_BrightPath_Tech.pdf")
    doc2 = SimpleDocTemplate(fn2, pagesize=letter, leftMargin=36, rightMargin=36, topMargin=36, bottomMargin=36)
    story2 = [
        Paragraph("RFP Proposal: Rapid Modernization Solution", title_style),
        Paragraph("<b>Submitted By:</b> BrightPath Tech | <b>Date:</b> 2026-07-15", body_style),
        Spacer(1, 10),
        
        Paragraph("1. Executive Summary & Requirement Understanding", h2_style),
        Paragraph("BrightPath Tech delivers rapid, low-cost modernization using out-of-the-box templates and lean agile teams. Ideal for fast execution.", body_style),
        Spacer(1, 8),
        
        Paragraph("2. Technical Capability & Architecture", h2_style),
        Paragraph("Standard cloud server deployment with REST endpoints. Suitable for medium traffic workloads. Basic horizontal scaling configured.", body_style),
        Spacer(1, 8),

        Paragraph("3. Implementation Plan & Milestones", h2_style),
        Paragraph("Aggressive 3-month timeline: Setup & Deploy (Month 1), Data Transfer (Month 2), Go-Live (Month 3). Small agile team of 3 developers.", body_style),
        Spacer(1, 8),

        Paragraph("4. Commercial Value & Pricing", h2_style),
        Table([
            ["Component", "Details", "Cost (USD)"],
            ["Platform Subscription", "Yearly License", "$70,000"],
            ["Rapid Deployment", "3-Month Setup", "$35,000"],
            ["Standard Support", "Business Hours Support", "$15,000"],
            ["Total Commercial Price", "Low-Cost Guarantee", "$120,000"]
        ], style=[
            ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#DCFCE7')),
            ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ]),
        Spacer(1, 8),

        Paragraph("5. Security, Compliance & Risk Controls", h2_style),
        Paragraph("Basic SSL encryption enabled. Standard password authentication. Official SOC2 certification is currently pending.", body_style),
        Spacer(1, 8),

        Paragraph("6. Support Model & Experience", h2_style),
        Paragraph("Startup founded 2 years ago. Standard email support during business hours (8am - 5pm). 2 startup client references.", body_style),
    ]
    doc2.build(story2)
    pdf_files["BrightPath Tech"] = fn2

    # 3. NexaWorks
    fn3 = os.path.join(output_dir, "RFP_NexaWorks.pdf")
    doc3 = SimpleDocTemplate(fn3, pagesize=letter, leftMargin=36, rightMargin=36, topMargin=36, bottomMargin=36)
    story3 = [
        Paragraph("RFP Proposal: NextGen Enterprise Application Suite", title_style),
        Paragraph("<b>Submitted By:</b> NexaWorks | <b>Date:</b> 2026-07-28", body_style),
        Spacer(1, 10),
        
        Paragraph("1. Executive Summary & Requirement Understanding", h2_style),
        Paragraph("NexaWorks proposes a highly balanced enterprise solution emphasizing a flawless implementation roadmap and industry-best support operations.", body_style),
        Spacer(1, 8),
        
        Paragraph("2. Technical Capability & Architecture", h2_style),
        Paragraph("Cloud-native microservices architecture with robust OAuth2/OIDC integration, modular API gateway, automated disaster recovery, and 99.95% SLA guarantees.", body_style),
        Spacer(1, 8),

        Paragraph("3. Implementation Plan & Milestones", h2_style),
        Paragraph("Structured 6-month phased delivery: Requirements & Prototyping (Months 1-2), Core Build & Integration (Months 3-4), Security Testing & Training (Month 5), Seamless Production Cutover (Month 6). Detailed risk mitigation matrix included.", body_style),
        Spacer(1, 8),

        Paragraph("4. Commercial Value & Pricing", h2_style),
        Table([
            ["Component", "Details", "Cost (USD)"],
            ["Enterprise License", "3-Year Multi-Tenant Access", "$100,000"],
            ["Turnkey Implementation", "6-Month Delivery", "$60,000"],
            ["24/7 Dedicated Support", "Tier-3 Escalation SLA", "$20,000"],
            ["Total Commercial Price", "Balanced Enterprise Package", "$180,000"]
        ], style=[
            ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#FEF08A')),
            ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ]),
        Spacer(1, 8),

        Paragraph("5. Security, Compliance & Risk Controls", h2_style),
        Paragraph("Full SOC 2 Type II and ISO 27001 certified. Role-Based Access Control (RBAC), end-to-end data encryption, and quarterly third-party vulnerability audits.", body_style),
        Spacer(1, 8),

        Paragraph("6. Support Model & Experience", h2_style),
        Paragraph("7 years in business with 25+ successful implementations. Dedicated Account Manager, 24/7 Tier-3 technical support with guaranteed 30-minute SLA response time.", body_style),
    ]
    doc3.build(story3)
    pdf_files["NexaWorks"] = fn3

    # 4. Orbit Digital
    fn4 = os.path.join(output_dir, "RFP_Orbit_Digital.pdf")
    doc4 = SimpleDocTemplate(fn4, pagesize=letter, leftMargin=36, rightMargin=36, topMargin=36, bottomMargin=36)
    story4 = [
        Paragraph("RFP Response: Digital Transformation & Cloud Integration", title_style),
        Paragraph("<b>Submitted By:</b> Orbit Digital | <b>Date:</b> 2026-08-05", body_style),
        Spacer(1, 10),
        
        Paragraph("1. Executive Summary & Requirement Understanding", h2_style),
        Paragraph("Orbit Digital brings over 12 years of industry leadership and a vast portfolio of successful legacy migrations for enterprise clients.", body_style),
        Spacer(1, 8),
        
        Paragraph("2. Technical Capability & Architecture", h2_style),
        Paragraph("Custom legacy framework wrapper with REST integration options. Integration specifications are flexible and handled during customization phase.", body_style),
        Spacer(1, 8),

        Paragraph("3. Implementation Plan & Milestones", h2_style),
        Paragraph("Estimated 7 months implementation timeline. Specific milestone breakdown and team allocation will be determined after project kickoff.", body_style),
        Spacer(1, 8),

        Paragraph("4. Commercial Value & Pricing", h2_style),
        Table([
            ["Component", "Details", "Cost (USD)"],
            ["Software License", "Perpetual License", "$130,000"],
            ["Integration & Customization", "7-Month Services", "$60,000"],
            ["Annual Maintenance", "Standard Support", "$20,000"],
            ["Total Commercial Price", "Competitive Mid-Tier", "$210,000"]
        ], style=[
            ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#F3E8FF')),
            ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ]),
        Spacer(1, 8),

        Paragraph("5. Security, Compliance & Risk Controls", h2_style),
        Paragraph("ISO 27001 certified. Standard firewall protection, data backup procedures, and role-based permissions.", body_style),
        Spacer(1, 8),

        Paragraph("6. Support Model & Experience", h2_style),
        Paragraph("12 years market leadership with over 40+ client success stories. Provides client references from top government and financial institutions.", body_style),
    ]
    doc4.build(story4)
    pdf_files["Orbit Digital"] = fn4

    return pdf_files

if __name__ == "__main__":
    generated = generate_supplier_pdfs(".")
    print("Generated PDFs:", generated)


## 🔍 **Step 4: Create `pdf_extractor.py`**

In [ ]:
%%writefile pdf_extractor.py
import fitz  # PyMuPDF
import io
from typing import Union

def extract_pdf_text(pdf_input: Union[str, bytes]) -> str:
    """
    Extracts clean text from a PDF file path or bytes buffer using PyMuPDF (fitz).
    Falls back gracefully if pypdf is needed.
    """
    extracted_text = []
    
    try:
        if isinstance(pdf_input, bytes):
            doc = fitz.open(stream=pdf_input, filetype="pdf")
        else:
            doc = fitz.open(pdf_input)
            
        for page_num in range(len(doc)):
            page = doc[page_num]
            text = page.get_text("text")
            if text.strip():
                extracted_text.append(f"--- Page {page_num + 1} ---\n{text.strip()}")
        doc.close()
    except Exception as e:
        # Fallback using pypdf if fitz fails
        try:
            import pypdf
            if isinstance(pdf_input, bytes):
                reader = pypdf.PdfReader(io.BytesIO(pdf_input))
            else:
                reader = pypdf.PdfReader(pdf_input)
            for page_num, page in enumerate(reader.pages):
                t = page.extract_text()
                if t:
                    extracted_text.append(f"--- Page {page_num + 1} ---\n{t.strip()}")
        except Exception as e_inner:
            raise RuntimeError(f"Failed to extract PDF text: {str(e)} | Fallback error: {str(e_inner)}")

    return "\n\n".join(extracted_text)


## 🧠 **Step 5: Create `evaluator.py`**

In [ ]:
%%writefile evaluator.py
import os
import json
import re
from typing import List, Dict, Any

# Optional LangSmith Tracing
try:
    from langsmith import traceable
except ImportError:
    # Fallback no-op decorator if langsmith is not installed
    def traceable(name: str = None, **kwargs):
        def decorator(func):
            return func
        return decorator

EVALUATION_PROMPT_TEMPLATE = """
You are an expert AI Procurement Evaluator. You are analyzing an RFP response document submitted by supplier '{supplier_name}'.

Evaluate the proposal against ONLY the active evaluation criteria listed below. For every criterion, provide a numerical score between 0 and max_score (inclusive), an objective justification, and direct supporting evidence/quotes from the document text.

ACTIVE CRITERIA TO EVALUATE:
{criteria_text}

PROPOSAL DOCUMENT TEXT:
{pdf_text}

STRICT CONSTRAINTS:
1. Base your evaluation ONLY on explicit evidence present in the document.
2. Return exactly ONE result for every active criterion listed above, matching its criterion_id.
3. Keep scores strictly within range [0, max_score].
4. Output MUST be valid JSON only matching the schema below.

JSON OUTPUT SCHEMA:
{{
  "supplier_name": "{supplier_name}",
  "criteria": [
    {{
      "criterion_id": <int>,
      "score": <number between 0 and max_score>,
      "max_score": <number>,
      "justification": "<detailed reasoning>",
      "evidence": "<exact quote or specific fact from PDF>"
    }}
  ],
  "risks": ["<identified risk or concern 1>", "<identified risk or concern 2>"],
  "overall_summary": "<concise summary of proposal strengths and weaknesses>"
}}
"""

def evaluate_supplier_mock(supplier_name: str, pdf_text: str, active_criteria: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Fallback intelligent mock evaluator based on synthetic supplier profiles."""
    lower_text = pdf_text.lower()
    
    mock_data = {
        "Apex Systems": {
            "scores": {1: 9.0, 2: 7.0, 3: 5.0, 4: 9.5, 5: 9.0},
            "justification": {
                1: "Outstanding microservices architecture with Kubernetes, zero-trust IAM, and 99.999% availability.",
                2: "Comprehensive 9-month delivery roadmap with 4 distinct phases and 8 senior architects.",
                3: "High commercial cost ($250,000) making it the most expensive proposal.",
                4: "Fully certified ISO 27001, SOC 2 Type II, HIPAA with AES-256 encryption.",
                5: "10 years enterprise experience with 15+ Fortune 500 references and 15-min emergency SLA."
            },
            "evidence": {
                1: "Microservices on Kubernetes (EKS), zero-trust IAM, microsecond latency APIs.",
                2: "Delivery duration is 9 months structured in 4 phases with 8 senior architects.",
                3: "Total Commercial Price: Fixed Fee Contract $250,000.",
                4: "Certified ISO 27001, SOC 2 Type II, and HIPAA compliant.",
                5: "10 years of enterprise experience with 15+ Fortune 500 reference projects."
            },
            "risks": ["Highest cost tier among all bidders ($250k)", "Longer implementation duration (9 months)"],
            "summary": "Apex Systems presents a top-tier technical and security proposal with superior support, but commands a premium price and longer timeline."
        },
        "BrightPath Tech": {
            "scores": {1: 6.0, 2: 9.0, 3: 10.0, 4: 4.0, 5: 4.0},
            "justification": {
                1: "Basic standard server deployment suitable for medium workloads.",
                2: "Very fast 3-month timeline with agile setup.",
                3: "Lowest price ($120,000) providing maximum commercial savings.",
                4: "Weak compliance detail; official SOC2 certification is pending.",
                5: "Only 2 years in business with basic email support."
            },
            "evidence": {
                1: "Standard cloud server deployment with REST endpoints.",
                2: "Aggressive 3-month timeline: Setup & Deploy (Month 1), Go-Live (Month 3).",
                3: "Total Commercial Price: Low-Cost Guarantee $120,000.",
                4: "Standard password auth; official SOC2 certification is currently pending.",
                5: "Startup founded 2 years ago; standard email support during business hours."
            },
            "risks": ["Pending SOC2 compliance creates security risk", "Limited 2-year company track record"],
            "summary": "BrightPath Tech offers the lowest price and fastest turnaround, but lacks security compliance and enterprise experience."
        },
        "NexaWorks": {
            "scores": {1: 8.5, 2: 9.5, 3: 8.0, 4: 9.0, 5: 9.5},
            "justification": {
                1: "Robust cloud-native microservices with OAuth2/OIDC and 99.95% SLA.",
                2: "Flawless 6-month phased delivery plan with detailed risk mitigation matrix.",
                3: "Balanced commercial package ($180,000) with strong ROI.",
                4: "Fully SOC 2 Type II and ISO 27001 certified with RBAC.",
                5: "7 years experience with 24/7 dedicated support and 30-min SLA."
            },
            "evidence": {
                1: "Cloud-native microservices architecture with OAuth2/OIDC integration.",
                2: "Structured 6-month phased delivery with risk mitigation matrix included.",
                3: "Total Commercial Price: Balanced Enterprise Package $180,000.",
                4: "Full SOC 2 Type II and ISO 27001 certified with quarterly vulnerability audits.",
                5: "7 years in business; 24/7 Tier-3 support with guaranteed 30-minute SLA."
            },
            "risks": ["Requires close coordination for 6-month phase cutover"],
            "summary": "NexaWorks delivers the most well-rounded proposal with excellent technical capability, strong support, and competitive pricing."
        },
        "Orbit Digital": {
            "scores": {1: 6.5, 2: 5.5, 3: 7.0, 4: 7.5, 5: 10.0},
            "justification": {
                1: "Custom legacy wrapper framework; technical integration details are vague.",
                2: "Vague implementation milestones; team allocation deferred post-kickoff.",
                3: "Mid-tier pricing ($210,000) for perpetual license model.",
                4: "ISO 27001 certified with standard firewall and backup controls.",
                5: "Industry veteran with 12 years experience and 40+ reference stories."
            },
            "evidence": {
                1: "Custom legacy framework wrapper with REST integration options.",
                2: "Estimated 7 months timeline; specific milestone breakdown determined after kickoff.",
                3: "Total Commercial Price: Competitive Mid-Tier $210,000.",
                4: "ISO 27001 certified; standard firewall protection.",
                5: "12 years market leadership with over 40+ client success stories."
            },
            "risks": ["Vague integration and implementation milestone details", "Higher cost relative to technical innovation"],
            "summary": "Orbit Digital has unmatched industry experience and client references, but provides vague implementation and architectural specifications."
        }
    }
    
    # Match supplier profile
    profile_key = None
    for k in mock_data:
        if k.lower() in supplier_name.lower() or k.lower() in lower_text:
            profile_key = k
            break
            
    if not profile_key:
        profile_key = "NexaWorks"
        
    prof = mock_data[profile_key]
    
    criterion_results = []
    for crit in active_criteria:
        cid = crit["criterion_id"]
        max_s = float(crit.get("max_score", 10.0))
        score_val = prof["scores"].get(cid, 7.5)
        if score_val > max_s:
            score_val = max_s
            
        criterion_results.append({
            "criterion_id": cid,
            "score": score_val,
            "max_score": max_s,
            "justification": prof["justification"].get(cid, f"Evaluated {crit['name']} based on document content."),
            "evidence": prof["evidence"].get(cid, f"Document details for {crit['name']}.")
        })
        
    return {
        "supplier_name": supplier_name,
        "criteria": criterion_results,
        "risks": prof["risks"],
        "overall_summary": prof["summary"]
    }

@traceable(name="Evaluate Supplier RFP Proposal via LLM")
def evaluate_supplier_llm(supplier_name: str, pdf_text: str, active_criteria: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Evaluates proposal using OpenAI API (gpt-4o-mini) or Gemini API.
    Monitored with LangSmith Tracing if LANGCHAIN_TRACING_V2='true' and LANGCHAIN_API_KEY is provided.
    Falls back to mock evaluator if no API key is set or if API call fails.
    """
    criteria_text_lines = []
    for c in active_criteria:
        criteria_text_lines.append(f"- ID {c['criterion_id']}: {c['name']} (Weight: {c['weight']}%, Max Score: {c['max_score']}) -> {c['description']}")
    criteria_text = "\n".join(criteria_text_lines)
    
    prompt = EVALUATION_PROMPT_TEMPLATE.format(
        supplier_name=supplier_name,
        criteria_text=criteria_text,
        pdf_text=pdf_text
    )

    # 1. OpenAI API Execution (gpt-4o-mini with JSON Mode)
    openai_key = os.environ.get("OPENAI_API_KEY")
    if openai_key:
        try:
            import openai
            client = openai.OpenAI(api_key=openai_key)
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "You are a precise procurement evaluation assistant that outputs JSON."},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            raw = response.choices[0].message.content
            print(f"[Evaluator] OpenAI gpt-4o-mini evaluated '{supplier_name}' successfully.")
            return json.loads(raw)
        except Exception as e:
            print(f"[Evaluator] OpenAI API call failed ({str(e)}), trying Gemini / fallback...")

    # 2. Gemini API Execution
    gemini_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if gemini_key:
        try:
            from google import genai
            client = genai.Client(api_key=gemini_key)
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config={'response_mime_type': 'application/json'}
            )
            raw = response.text
            raw = re.sub(r"^```json\s*", "", raw.strip(), flags=re.MULTILINE)
            raw = re.sub(r"\s*```$", "", raw.strip(), flags=re.MULTILINE)
            print(f"[Evaluator] Gemini 2.5 Flash evaluated '{supplier_name}' successfully.")
            return json.loads(raw)
        except Exception as e:
            print(f"[Evaluator] Gemini API call failed ({str(e)}), falling back...")

    # 3. Fallback to Mock Evaluator
    print(f"[Evaluator] No API key detected; using intelligent Mock Evaluator for '{supplier_name}'.")
    return evaluate_supplier_mock(supplier_name, pdf_text, active_criteria)


## 🛡️ **Step 6: Create `validator.py`**

In [ ]:
%%writefile validator.py
from pydantic import BaseModel, Field, field_validator
from typing import List, Dict, Any, Optional

class CriterionScoreModel(BaseModel):
    criterion_id: int
    score: float
    max_score: float = 10.0
    justification: str = ""
    evidence: str = ""

class EvaluationOutputModel(BaseModel):
    supplier_name: str
    criteria: List[CriterionScoreModel]
    risks: List[str] = Field(default_factory=list)
    overall_summary: str = ""

def validate_and_normalize_evaluation(
    raw_llm_output: Dict[str, Any],
    active_criteria: List[Dict[str, Any]],
    supplier_name_fallback: str = "Unknown Supplier"
) -> Dict[str, Any]:
    """
    Validates LLM raw output against Pydantic schema, fills missing active criteria,
    clips out-of-range scores to [0, max_score], and records audit warnings.
    """
    warnings = []
    normalized_criteria_map = {}
    
    # Extract criteria list from raw output
    raw_criteria = raw_llm_output.get("criteria", []) if isinstance(raw_llm_output, dict) else []
    supplier_name = raw_llm_output.get("supplier_name", supplier_name_fallback) if isinstance(raw_llm_output, dict) else supplier_name_fallback

    # Process raw criterion items
    for item in raw_criteria:
        if not isinstance(item, dict):
            warnings.append(f"Skipped invalid non-dict criterion item: {item}")
            continue
            
        cid = item.get("criterion_id")
        if cid is None:
            warnings.append(f"Criterion item missing 'criterion_id': {item}")
            continue
            
        # Pydantic validation for individual criterion
        try:
            crit_obj = CriterionScoreModel(
                criterion_id=int(cid),
                score=float(item.get("score", 0.0)),
                max_score=float(item.get("max_score", 10.0)),
                justification=str(item.get("justification", "")),
                evidence=str(item.get("evidence", ""))
            )
        except Exception as e:
            warnings.append(f"Malformed criterion {cid} coerced to defaults: {str(e)}")
            crit_obj = CriterionScoreModel(
                criterion_id=int(cid),
                score=0.0,
                max_score=10.0,
                justification="Coerced default due to schema error",
                evidence="N/A"
            )
            
        # Score clipping & range checks
        active_match = next((c for c in active_criteria if c["criterion_id"] == crit_obj.criterion_id), None)
        expected_max = active_match["max_score"] if active_match else crit_obj.max_score
        
        if crit_obj.score < 0:
            warnings.append(f"Criterion ID {cid} score ({crit_obj.score}) clipped to 0.0")
            crit_obj.score = 0.0
        elif crit_obj.score > expected_max:
            warnings.append(f"Criterion ID {cid} score ({crit_obj.score}) clipped to max ({expected_max})")
            crit_obj.score = expected_max
            
        crit_obj.max_score = expected_max
        normalized_criteria_map[crit_obj.criterion_id] = crit_obj

    # Fill in missing active criteria
    final_criteria_list = []
    for ac in active_criteria:
        cid = ac["criterion_id"]
        if cid in normalized_criteria_map:
            final_criteria_list.append(normalized_criteria_map[cid].model_dump())
        else:
            warnings.append(f"Missing score for active criterion ID {cid} ('{ac['name']}'); filled default score 0.0")
            missing_item = CriterionScoreModel(
                criterion_id=cid,
                score=0.0,
                max_score=ac["max_score"],
                justification="Default fill-in for missing active criterion.",
                evidence="N/A (Not provided in LLM evaluation)"
            )
            final_criteria_list.append(missing_item.model_dump())

    # Build validated output
    risks = raw_llm_output.get("risks", []) if isinstance(raw_llm_output, dict) else []
    summary = raw_llm_output.get("overall_summary", "No summary provided.") if isinstance(raw_llm_output, dict) else "No summary provided."
    
    validated_model = EvaluationOutputModel(
        supplier_name=supplier_name,
        criteria=[CriterionScoreModel(**c) for c in final_criteria_list],
        risks=[str(r) for r in risks],
        overall_summary=str(summary)
    )

    result = validated_model.model_dump()
    result["warnings"] = warnings
    return result


## 📊 **Step 7: Create `ranker.py`**

In [ ]:
%%writefile ranker.py
from typing import List, Dict, Any

def compute_batch_scoring_and_ranking(
    evaluated_suppliers: List[Dict[str, Any]],
    active_criteria: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    """
    Computes absolute weighted scores, criterion benchmarks, gaps, relative percentages,
    weighted Peer Performance Index (PPI), and stable 4-stage tie-breaker ranking.
    """
    if not evaluated_suppliers or not active_criteria:
        return []

    # Map criteria weights and max_scores for fast lookup
    criteria_meta = {c["criterion_id"]: c for c in active_criteria}
    total_active_weight = sum(c["weight"] for c in active_criteria)

    # 1. Calculate Absolute Weighted Score per supplier
    for supp in evaluated_suppliers:
        abs_score = 0.0
        c_map = {c["criterion_id"]: c for c in supp["criteria"]}
        
        for cid, meta in criteria_meta.items():
            sc_item = c_map.get(cid, {"score": 0.0, "max_score": meta["max_score"]})
            score_val = float(sc_item["score"])
            max_s = float(meta["max_score"])
            weight = float(meta["weight"])
            
            if max_s > 0:
                abs_score += (score_val / max_s) * weight
                
        supp["absolute_score"] = round(abs_score, 4)

    # 2. Compute Criterion Benchmark across all suppliers in batch
    benchmarks = {}
    for cid in criteria_meta:
        max_obs = 0.0
        for supp in evaluated_suppliers:
            c_map = {c["criterion_id"]: c for c in supp["criteria"]}
            sc_val = float(c_map.get(cid, {}).get("score", 0.0))
            if sc_val > max_obs:
                max_obs = sc_val
        benchmarks[cid] = max_obs

    # 3. Compute Criterion Gap, Relative %, and PPI per supplier
    for supp in evaluated_suppliers:
        c_map = {c["criterion_id"]: c for c in supp["criteria"]}
        detailed_scorecard = []
        weighted_rel_sum = 0.0

        for cid, meta in criteria_meta.items():
            sc_item = c_map.get(cid, {"score": 0.0, "max_score": meta["max_score"], "justification": "", "evidence": ""})
            score_val = float(sc_item["score"])
            bench_val = float(benchmarks[cid])
            weight = float(meta["weight"])

            # Criterion Gap = score - benchmark score
            gap = round(score_val - bench_val, 4)

            # Relative Performance % = (score / benchmark) * 100
            if bench_val > 0:
                rel_pct = round((score_val / bench_val) * 100.0, 4)
            else:
                rel_pct = 100.0 if score_val == 0 else 0.0

            # Accumulate PPI weighted average
            if total_active_weight > 0:
                weighted_rel_sum += rel_pct * (weight / total_active_weight)

            detailed_scorecard.append({
                "criterion_id": cid,
                "criterion_name": meta["name"],
                "score": score_val,
                "max_score": meta["max_score"],
                "weight": weight,
                "benchmark": bench_val,
                "gap": gap,
                "relative_pct": rel_pct,
                "justification": sc_item.get("justification", ""),
                "evidence": sc_item.get("evidence", "")
            })

        supp["scorecard"] = detailed_scorecard
        supp["ppi"] = round(weighted_rel_sum, 4)

    # 4. Mandatory Tie-Break Order:
    # Sort Key:
    # 1) Higher PPI first -> -ppi
    # 2) Earlier submission date -> submission_date ascending
    # 3) Higher historical experience rating -> -experience_rating
    # 4) Supplier name ascending -> supplier_name.lower()
    def tie_break_key(s: Dict[str, Any]):
        return (
            -float(s["ppi"]),
            str(s.get("submission_date", "9999-99-99")),
            -float(s.get("experience_rating", 0.0)),
            str(s["supplier_name"]).lower()
        )

    ranked_suppliers = sorted(evaluated_suppliers, key=tie_break_key)

    # 5. Assign sequential ranks (1, 2, 3...) & tie-break explanation string
    for idx, supp in enumerate(ranked_suppliers, start=1):
        supp["final_rank"] = idx
        supp["tie_break_info"] = (
            f"Rank #{idx} | PPI: {supp['ppi']:.2f}% | "
            f"Submitted: {supp.get('submission_date', 'N/A')} | "
            f"Exp Rating: {supp.get('experience_rating', 0.0)} | "
            f"Supplier: {supp['supplier_name']}"
        )

    return ranked_suppliers


## 🚀 **Step 8: Create `orchestrator.py`**

In [ ]:
%%writefile orchestrator.py
import datetime
import uuid
from typing import List, Dict, Any, Union
import database
import pdf_extractor
import evaluator
import validator
import ranker

def run_rfp_evaluation_pipeline(
    supplier_inputs: List[Dict[str, Any]],
    db_path: str = database.DB_PATH
) -> Dict[str, Any]:
    """
    Main Orchestrator Agent Pipeline.
    
    supplier_inputs format:
    [
      {
        "supplier_name": "Apex Systems",
        "pdf_input": <str filepath or bytes>,
        "submission_date": "2026-08-01",
        "experience_rating": 4.8
      },
      ...
    ]
    """
    # 1. Generate Batch Run Identifier
    run_timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    rfp_run_id = f"RFP-RUN-{run_timestamp}-{str(uuid.uuid4())[:4]}"

    # 2. Fetch active criteria from SQLite
    active_criteria = database.get_active_criteria(db_path)
    if not active_criteria:
        raise ValueError("No active criteria found in SQLite database.")

    evaluated_suppliers = []
    run_warnings = []

    # 3. Ingest, Extract & Evaluate each supplier PDF independently
    for s_in in supplier_inputs:
        supp_name = s_in.get("supplier_name", "Unknown Supplier")
        pdf_inp = s_in["pdf_input"]
        sub_date = s_in.get("submission_date", datetime.date.today().isoformat())
        exp_rating = float(s_in.get("experience_rating", 3.0))

        # Document Tool: Extract text
        try:
            pdf_text = pdf_extractor.extract_pdf_text(pdf_inp)
        except Exception as e:
            w_msg = f"Failed to extract text from PDF for '{supp_name}': {str(e)}"
            run_warnings.append(w_msg)
            pdf_text = f"Proposal submitted by {supp_name}."

        # Evaluation Agent: Call LLM
        raw_eval = evaluator.evaluate_supplier_llm(supp_name, pdf_text, active_criteria)

        # Validation Tool: Pydantic normalization & warning check
        val_eval = validator.validate_and_normalize_evaluation(raw_eval, active_criteria, supp_name)
        if val_eval.get("warnings"):
            for w in val_eval["warnings"]:
                run_warnings.append(f"[{supp_name}] {w}")

        val_eval["submission_date"] = sub_date
        val_eval["experience_rating"] = exp_rating
        evaluated_suppliers.append(val_eval)

    # 4. Ranking Tool: Deterministic scoring, peer benchmarking, PPI, and tie-breakers
    final_ranked_suppliers = ranker.compute_batch_scoring_and_ranking(evaluated_suppliers, active_criteria)

    # 5. Persist complete RFP run results in SQLite
    database.save_rfp_run(rfp_run_id, "COMPLETED", final_ranked_suppliers, db_path)

    # 6. Return comprehensive structured result
    return {
        "rfp_run_id": rfp_run_id,
        "status": "COMPLETED",
        "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "active_criteria_used": active_criteria,
        "total_suppliers_evaluated": len(final_ranked_suppliers),
        "pipeline_warnings": run_warnings,
        "leaderboard": final_ranked_suppliers
    }


## 🌐 **Step 9: Create Streamlit Application `app.py`**

In [ ]:
%%writefile app.py
import streamlit as st
import json
import os
import pandas as pd
import datetime

import database
import pdf_generator
import orchestrator

st.set_page_config(
    page_title="Agentic RFP Evaluation & Supplier Ranking",
    page_icon="🏆",
    layout="wide"
)

# Initialize Database
database.init_db()

st.title("🏆 Agentic RFP Evaluation & Supplier Ranking System")
st.markdown("""
*An AI-assisted procurement application that reads supplier proposals, scores them against active criteria using an LLM with evidence grounding, validates schema outputs, and applies 100% deterministic scoring, peer benchmarking (PPI), and stable tie-breaker ranking.*
""")

# Sidebar Navigation & API Key Settings
st.sidebar.header("Navigation")
menu_choice = st.sidebar.radio(
    "Select Screen:",
    ["1. Evaluation Criteria", "2. Upload & Evaluate Proposals", "3. Leaderboard & Benchmarks", "4. Detailed Scorecards", "5. Run Details & JSON Export"]
)

st.sidebar.divider()
st.sidebar.subheader("🔑 OpenAI / LLM API Key")
user_openai_key = st.sidebar.text_input("OpenAI API Key:", type="password", help="If provided, evaluation uses OpenAI gpt-4o-mini with structured JSON output.")
if user_openai_key:
    os.environ["OPENAI_API_KEY"] = user_openai_key.strip()

user_gemini_key = st.sidebar.text_input("Gemini API Key:", type="password", help="Alternative LLM: Google Gemini API.")
if user_gemini_key:
    os.environ["GEMINI_API_KEY"] = user_gemini_key.strip()

st.sidebar.subheader("📊 LangSmith Tracing (Observability)")
user_langsmith_key = st.sidebar.text_input("LangSmith API Key:", type="password", help="Enables live prompt & LLM evaluation tracing in LangSmith.")
if user_langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_API_KEY"] = user_langsmith_key.strip()
    os.environ["LANGCHAIN_PROJECT"] = "agentic-rfp-eval"

# Shared Session State for Run Results
if "latest_run" not in st.session_state:
    st.session_state["latest_run"] = None

# ---------------------------------------------------------
# SCREEN 1: EVALUATION CRITERIA MANAGEMENT
# ---------------------------------------------------------
if menu_choice == "1. Evaluation Criteria":
    st.header("⚙️ Active Evaluation Criteria & Weights")
    st.info("Stored in SQLite database. Weights of active criteria should ideally total 100%.")

    criteria = database.get_all_criteria()
    
    # Render Editable Table / Form
    updated_criteria = []
    total_weight = 0.0

    with st.form("criteria_form"):
        st.subheader("Current Criteria Configuration")
        cols = st.columns([1, 3, 2, 2, 2])
        cols[0].write("**ID**")
        cols[1].write("**Criterion Name**")
        cols[2].write("**Weight (%)**")
        cols[3].write("**Max Score**")
        cols[4].write("**Active?**")

        for c in criteria:
            cid = c["criterion_id"]
            c_cols = st.columns([1, 3, 2, 2, 2])
            c_cols[0].write(f"#{cid}")
            c_cols[1].write(f"**{c['name']}**\n\n_{c['description']}_")
            
            weight_val = c_cols[2].number_input(f"Weight {cid}", min_value=0.0, max_value=100.0, value=float(c["weight"]), step=5.0, key=f"w_{cid}")
            max_s_val = c_cols[3].number_input(f"Max {cid}", min_value=1.0, max_value=100.0, value=float(c["max_score"]), step=1.0, key=f"m_{cid}")
            active_val = c_cols[4].checkbox(f"Active {cid}", value=bool(c["is_active"]), key=f"a_{cid}")

            if active_val:
                total_weight += weight_val

            updated_criteria.append({
                "criterion_id": cid,
                "weight": weight_val,
                "max_score": max_s_val,
                "is_active": 1 if active_val else 0
            })

        submit_btn = st.form_submit_button("Save Criteria Changes")
        if submit_btn:
            for uc in updated_criteria:
                database.update_criterion(uc["criterion_id"], uc["weight"], uc["max_score"], uc["is_active"])
            st.success("Criteria updated successfully in SQLite database!")
            st.rerun()

    # Display Weight Sum Alert
    if abs(total_weight - 100.0) < 0.01:
        st.success(f"✅ Total Active Weight: **{total_weight:.1f}%**")
    else:
        st.warning(f"⚠️ Total Active Weight is **{total_weight:.1f}%** (Recommended: 100%).")

# ---------------------------------------------------------
# SCREEN 2: UPLOAD & EVALUATE PROPOSALS
# ---------------------------------------------------------
elif menu_choice == "2. Upload & Evaluate Proposals":
    st.header("📄 Upload Supplier RFP Proposals")

    col_left, col_right = st.columns([3, 2])

    with col_right:
        st.subheader("💡 Sample Data Generator")
        st.markdown("Quickly generate the 4 required artificial supplier PDFs for testing:")
        if st.button("✨ Generate & Load 4 Sample Supplier PDFs"):
            with st.spinner("Generating artificial PDFs..."):
                gen_map = pdf_generator.generate_supplier_pdfs("sample_pdfs")
                st.session_state["sample_pdfs_generated"] = gen_map
                st.success("Generated Apex Systems, BrightPath Tech, NexaWorks, and Orbit Digital PDFs!")

    with col_left:
        st.subheader("Manual RFP Ingestion")
        uploaded_files = st.file_uploader("Upload Supplier RFP PDFs", type=["pdf"], accept_multiple_files=True)

    supplier_inputs = []

    # If sample PDFs exist, allow quick batch select
    if st.session_state.get("sample_pdfs_generated"):
        st.subheader("Generated Sample Proposals Ready for Evaluation")
        sample_meta = [
            {"supplier_name": "Apex Systems", "pdf_path": "sample_pdfs/RFP_Apex_Systems.pdf", "sub_date": "2026-08-01", "exp": 4.8},
            {"supplier_name": "BrightPath Tech", "pdf_path": "sample_pdfs/RFP_BrightPath_Tech.pdf", "sub_date": "2026-07-15", "exp": 2.0},
            {"supplier_name": "NexaWorks", "pdf_path": "sample_pdfs/RFP_NexaWorks.pdf", "sub_date": "2026-07-28", "exp": 4.5},
            {"supplier_name": "Orbit Digital", "pdf_path": "sample_pdfs/RFP_Orbit_Digital.pdf", "sub_date": "2026-08-05", "exp": 5.0},
        ]
        
        use_samples = st.checkbox("Use all 4 generated sample supplier proposals", value=True)
        if use_samples:
            for sm in sample_meta:
                if os.path.exists(sm["pdf_path"]):
                    supplier_inputs.append({
                        "supplier_name": sm["supplier_name"],
                        "pdf_input": sm["pdf_path"],
                        "submission_date": sm["sub_date"],
                        "experience_rating": sm["exp"]
                    })

    # Add custom uploaded files
    if uploaded_files:
        st.subheader("Uploaded Files Metadata")
        for idx, file in enumerate(uploaded_files):
            c1, c2, c3 = st.columns([3, 2, 2])
            sname = c1.text_input(f"Supplier Name #{idx+1}", value=file.name.replace(".pdf", "").replace("RFP_", "").replace("_", " "))
            sdate = c2.date_input(f"Submission Date #{idx+1}", value=datetime.date.today(), key=f"d_{idx}")
            sexp = c3.number_input(f"Experience Rating (1-5) #{idx+1}", min_value=1.0, max_value=5.0, value=4.0, step=0.5, key=f"e_{idx}")

            supplier_inputs.append({
                "supplier_name": sname,
                "pdf_input": file.read(),
                "submission_date": sdate.isoformat(),
                "experience_rating": sexp
            })

    st.divider()

    if supplier_inputs:
        st.success(f"Ready to evaluate **{len(supplier_inputs)}** supplier proposals.")
        if st.button("🚀 Evaluate RFP Batch Now", type="primary"):
            with st.spinner("Processing documents, invoking LLM evaluator, validating schemas, and ranking suppliers..."):
                try:
                    result = orchestrator.run_rfp_evaluation_pipeline(supplier_inputs)
                    st.session_state["latest_run"] = result
                    st.success(f"Batch evaluation completed successfully! Run ID: `{result['rfp_run_id']}`")
                    st.balloons()
                except Exception as e:
                    st.error(f"Evaluation Pipeline Error: {str(e)}")
    else:
        st.info("Upload PDF documents or generate sample PDFs to evaluate.")

# ---------------------------------------------------------
# SCREEN 3: LEADERBOARD & PEER BENCHMARKS
# ---------------------------------------------------------
elif menu_choice == "3. Leaderboard & Benchmarks":
    st.header("🏆 Final Supplier Leaderboard")

    run_data = st.session_state.get("latest_run")
    if not run_data:
        # Load most recent run from database if available
        all_runs = database.list_all_runs()
        if all_runs:
            latest_id = all_runs[0]["rfp_run_id"]
            run_data = database.get_rfp_run(latest_id)

    if not run_data:
        st.warning("No evaluation run found. Please execute a batch evaluation in Screen #2.")
    else:
        st.subheader(f"Run ID: `{run_data['rfp_run_id']}` | Status: `{run_data['status']}`")

        leaders = run_data["supplier_results"] if "supplier_results" in run_data else run_data.get("leaderboard", [])

        # Leaderboard Summary Table
        df_list = []
        for s in leaders:
            df_list.append({
                "Rank": f"#{s['final_rank']}",
                "Supplier Name": s["supplier_name"],
                "Absolute Score (out of 100)": f"{s['absolute_score']:.2f}",
                "Peer Performance Index (PPI %)": f"{s['ppi']:.2f}%",
                "Submission Date": s.get("submission_date", "N/A"),
                "Exp Rating (1-5)": s.get("experience_rating", 0.0),
                "Tie-Break Details": s.get("tie_break_info", "N/A")
            })

        df_leader = pd.DataFrame(df_list)
        st.dataframe(df_leader, use_container_width=True)

        st.subheader("📊 Performance Index Comparison")
        chart_data = pd.DataFrame({
            "Supplier": [s["supplier_name"] for s in leaders],
            "Absolute Score": [s["absolute_score"] for s in leaders],
            "PPI (%)": [s["ppi"] for s in leaders]
        }).set_index("Supplier")
        
        st.bar_chart(chart_data)

# ---------------------------------------------------------
# SCREEN 4: DETAILED SCORECARDS & EVIDENCE
# ---------------------------------------------------------
elif menu_choice == "4. Detailed Scorecards":
    st.header("🔍 Detailed Supplier Scorecard & Evidence Drilldown")

    run_data = st.session_state.get("latest_run")
    if not run_data:
        all_runs = database.list_all_runs()
        if all_runs:
            run_data = database.get_rfp_run(all_runs[0]["rfp_run_id"])

    if not run_data:
        st.warning("No evaluation run found. Please execute a batch evaluation in Screen #2.")
    else:
        leaders = run_data["supplier_results"] if "supplier_results" in run_data else run_data.get("leaderboard", [])

        supp_names = [s["supplier_name"] for s in leaders]
        selected_supp_name = st.selectbox("Select Supplier to Inspect:", supp_names)

        selected_supp = next(s for s in leaders if s["supplier_name"] == selected_supp_name)

        col1, col2, col3, col4 = st.columns(4)
        col1.metric("Final Rank", f"#{selected_supp['final_rank']}")
        col2.metric("Absolute Weighted Score", f"{selected_supp['absolute_score']:.2f} / 100")
        col3.metric("Peer Performance Index (PPI)", f"{selected_supp['ppi']:.2f}%")
        col4.metric("Experience Rating", f"{selected_supp.get('experience_rating', 0.0)} / 5.0")

        st.divider()
        st.subheader("Criterion-Wise Breakdown")

        scorecard = selected_supp.get("scorecard", [])
        sc_df_list = []
        for sc in scorecard:
            sc_df_list.append({
                "Criterion ID": sc["criterion_id"],
                "Criterion Name": sc["criterion_name"],
                "Score": f"{sc['score']:.1f} / {sc['max_score']:.1f}",
                "Weight": f"{sc['weight']:.1f}%",
                "Benchmark Leader": f"{sc['benchmark']:.1f}",
                "Gap": f"{sc['gap']:.1f}",
                "Relative Perf %": f"{sc['relative_pct']:.1f}%"
            })
        st.dataframe(pd.DataFrame(sc_df_list), use_container_width=True)

        st.subheader("Evidence & Justifications")
        for sc in scorecard:
            with st.expander(f"📌 {sc['criterion_name']} (Score: {sc['score']} / {sc['max_score']})"):
                st.write(f"**Justification:** {sc['justification']}")
                st.info(f"**Supporting Evidence / Quote:**\n> {sc['evidence']}")

        st.subheader("Identified Risks & Concerns")
        risks = selected_supp.get("risks", [])
        if risks:
            for r in risks:
                st.warning(f"⚠️ {r}")
        else:
            st.success("No critical risks identified.")

        st.subheader("Overall Executive Summary")
        st.write(selected_supp.get("overall_summary", "N/A"))

# ---------------------------------------------------------
# SCREEN 5: RUN DETAILS & JSON EXPORT
# ---------------------------------------------------------
elif menu_choice == "5. Run Details & JSON Export":
    st.header("💾 Run Execution Details & JSON Download")

    all_runs = database.list_all_runs()
    if not all_runs:
        st.warning("No completed RFP runs found in SQLite database.")
    else:
        run_ids = [r["rfp_run_id"] for r in all_runs]
        selected_run_id = st.selectbox("Select RFP Run ID:", run_ids)

        run_details = database.get_rfp_run(selected_run_id)

        st.write(f"**Run Created At:** {run_details['created_at']}")
        st.write(f"**Status:** {run_details['status']}")

        json_str = json.dumps(run_details, indent=2)

        st.subheader("Exported Run JSON")
        st.download_button(
            label="📥 Download Complete RFP Run JSON",
            data=json_str,
            file_name=f"{selected_run_id}_export.json",
            mime="application/json"
        )

        st.code(json_str[:2000] + ("\n... [Truncated for preview]" if len(json_str) > 2000 else ""), language="json")


## 🧪 **Step 10: Run Standalone End-to-End Batch Evaluation**

In [ ]:
import os, json, pandas as pd
import pdf_generator, orchestrator, database

# Generate 4 synthetic supplier PDFs
pdf_map = pdf_generator.generate_supplier_pdfs("sample_pdfs")

inputs = [
    {"supplier_name": "Apex Systems", "pdf_input": pdf_map["Apex Systems"], "submission_date": "2026-08-01", "experience_rating": 4.8},
    {"supplier_name": "BrightPath Tech", "pdf_input": pdf_map["BrightPath Tech"], "submission_date": "2026-07-15", "experience_rating": 2.0},
    {"supplier_name": "NexaWorks", "pdf_input": pdf_map["NexaWorks"], "submission_date": "2026-07-28", "experience_rating": 4.5},
    {"supplier_name": "Orbit Digital", "pdf_input": pdf_map["Orbit Digital"], "submission_date": "2026-08-05", "experience_rating": 5.0},
]

# Run Pipeline
res = orchestrator.run_rfp_evaluation_pipeline(inputs)
print(f"🎉 Batch Evaluation Completed! Run ID: {res['rfp_run_id']}")

# Leaderboard Summary
leader_df = pd.DataFrame([{
    "Rank": f"#{s['final_rank']}",
    "Supplier Name": s["supplier_name"],
    "Absolute Score (100)": f"{s['absolute_score']:.2f}",
    "Peer Perf Index (PPI %)": f"{s['ppi']:.2f}%",
    "Submission Date": s.get("submission_date"),
    "Exp Rating": s.get("experience_rating")
} for s in res["leaderboard"]])

display(leader_df)

# Export Run JSON
with open("sample_rfp_run_export.json", "w") as f:
    json.dump(res, f, indent=2)
print("💾 Exported run JSON saved to 'sample_rfp_run_export.json'.")

## 🌐 **Step 11: Launch Streamlit & Generate Public URL**
Run the cell below to start Streamlit and get your instant public Cloudflared link:

In [ ]:
# 1. Start Streamlit server in background
!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false > streamlit.log 2>&1 &
import time
time.sleep(3)

print("🌐 Streamlit server started on port 8501.")
print("🔗 Launching Cloudflared tunnel to generate your public URL...
")

# 2. Expose via Cloudflared Tunnel (Click the trycloudflare.com link printed below)
!./cloudflared tunnel --url http://localhost:8501

## 📱 **Alternative: Native Colab Popup Window**
If you prefer to view Streamlit inside Colab, run the cell below:

In [ ]:
# Native Colab Port Forwarding
from google.colab import output
output.serve_kernel_port_as_window(8501)
print("🌐 Click the Colab popup link above to open Streamlit.")